# 🧠 InceptionV3 Hap Sınıflandırma — Eğitim & Kayıt
**Amaç:** InceptionV3 modelini eğit, `Hap_Modelleri_V3` klasörüne kaydet.
**Giriş boyutu:** 299×299 | **Preprocessing:** [-1, 1] normalize

In [ ]:
# ── HÜCRE 1: Drive Bağlantısı ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive bağlandı!')

Mounted at /content/drive
✅ Drive bağlandı!


In [ ]:
# ── HÜCRE 2: Kütüphaneler ──────────────────────────────────────────────────
import os, cv2, numpy as np, random, warnings
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

warnings.filterwarnings('ignore')
print('✅ Kütüphaneler yüklendi!')

✅ Kütüphaneler yüklendi!


In [ ]:
# ── HÜCRE 3: Veri Yolu ve Sınıflar ────────────────────────────────────────
veri_yolu = "/content/drive/MyDrive/YapayZeka_Hap_Projesi/archive/Drug Vision/Data Combined"
siniflar  = sorted([s for s in os.listdir(veri_yolu)
                    if os.path.isdir(os.path.join(veri_yolu, s))])

print(f"✅ Toplam {len(siniflar)} İlaç Sınıfı:")
for i, s in enumerate(siniflar):
    print(f"  [{i}] {s}")

✅ Toplam 10 İlaç Sınıfı:
  [0] Alaxan
  [1] Bactidol
  [2] Bioflu
  [3] Biogesic
  [4] DayZinc
  [5] Decolgen
  [6] Fish Oil
  [7] Kremil S
  [8] Medicol
  [9] Neozep


In [ ]:
# ── HÜCRE 4-5-6 YERİNE: Bellek Dostu Veri Yükleme (tf.data) ───────────
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

BOYUT = (299, 299)
BATCH = 32

print("📂 Veriler Drive'dan batch'ler halinde okunacak şekilde ayarlanıyor...")

# 1. Eğitim Verisetini Oluştur
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    veri_yolu,
    validation_split=0.2,
    subset="training",
    seed=9751,
    image_size=BOYUT,
    batch_size=BATCH,
    label_mode='categorical'
)

# 2. Doğrulama (Test) Verisetini Oluştur
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    veri_yolu,
    validation_split=0.2,
    subset="validation",
    seed=9751,
    image_size=BOYUT,
    batch_size=BATCH,
    label_mode='categorical'
)

# 3. InceptionV3 Preprocessing İşlemi ([0, 255] aralığını [-1, 1] yapar)
def preprocess(image, label):
    return preprocess_input(image), label

train_ds = train_ds_raw.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds_raw.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# 4. Performans için Prefetch ekle
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

print('✅ Pipeline hazır ve bellek çökmesi önlendi!')

📂 Veriler Drive'dan batch'ler halinde okunacak şekilde ayarlanıyor...
Found 10000 files belonging to 10 classes.
Using 8000 files for training.
Found 10000 files belonging to 10 classes.
Using 2000 files for validation.
✅ Pipeline hazır ve bellek çökmesi önlendi!


In [ ]:
# ── HÜCRE 7: InceptionV3 Model Mimarisi ───────────────────────────────────
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False  # İlk aşamada tamamen dondur

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(len(siniflar), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'✅ InceptionV3 modeli hazır.')
print(f'   Eğitilebilir parametre: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ InceptionV3 modeli hazır.
   Eğitilebilir parametre: 1,187,082


In [ ]:
# ── HÜCRE 8: Aşama 1 — Sadece Baş Katmanları Eğit ─────────────────────────
import os
kayit_yolu = '/content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5'
os.makedirs(os.path.dirname(kayit_yolu), exist_ok=True)

callbacks_1 = [
    EarlyStopping(patience=5, restore_best_weights=True,
                  monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6,
                     monitor='val_loss', verbose=1),
    ModelCheckpoint(kayit_yolu, save_best_only=True,
                    monitor='val_accuracy', verbose=1)
]

print('🚀 Aşama 1 — Transfer Learning Başlıyor...')
history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_1,
    verbose=1
)
print(f'✅ Aşama 1 tamamlandı! En iyi val_accuracy: %{max(history_1.history["val_accuracy"])*100:.2f}')

🚀 Aşama 1 — Transfer Learning Başlıyor...
Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4477 - loss: 1.7591
Epoch 1: val_accuracy improved from None to 0.69250, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 646s 3s/step - accuracy: 0.5447 - loss: 1.3897 - val_accuracy: 0.6925 - val_loss: 0.9284 - learning_rate: 0.0010
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6739 - loss: 0.9409
Epoch 2: val_accuracy improved from 0.69250 to 0.72300, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 597s 2s/step - accuracy: 0.6875 - loss: 0.9109 - val_accuracy: 0.7230 - val_loss: 0.7644 - learning_rate: 0.0010
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7300 - loss: 0.7467
Epoch 3: val_accuracy improved from 0.72300 to 0.74550, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 3: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 587s 2s/step - accuracy: 0.7360 - loss: 0.7433 - val_accuracy: 0.7455 - val_loss: 0.7325 - learning_rate: 0.0010
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7572 - loss: 0.6872
Epoch 4: val_accuracy improved from 0.74550 to 0.76150, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 4: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 588s 2s/step - accuracy: 0.7609 - loss: 0.6845 - val_accuracy: 0.7615 - val_loss: 0.6868 - learning_rate: 0.0010
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7966 - loss: 0.5873
Epoch 5: val_accuracy improved from 0.76150 to 0.76700, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5


...


Epoch 6: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 591s 2s/step - accuracy: 0.8085 - loss: 0.5427 - val_accuracy: 0.7745 - val_loss: 0.6733 - learning_rate: 0.0010
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8249 - loss: 0.5067
Epoch 7: val_accuracy did not improve from 0.77450
250/250 ━━━━━━━━━━━━━━━━━━━━ 589s 2s/step - accuracy: 0.8236 - loss: 0.5090 - val_accuracy: 0.7695 - val_loss: 0.6771 - learning_rate: 0.0010
Epoch 8/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8412 - loss: 0.4449
Epoch 8: val_accuracy did not improve from 0.77450
250/250 ━━━━━━━━━━━━━━━━━━━━ 589s 2s/step - accuracy: 0.8380 - loss: 0.4655 - val_accuracy: 0.7635 - val_loss: 0.7034 - learning_rate: 0.0010
Epoch 9/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8628 - loss: 0.4111
Epoch 9: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.

Epoch 9: val_accuracy did not improve f


Epoch 10: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 585s 2s/step - accuracy: 0.8895 - loss: 0.3223 - val_accuracy: 0.7950 - val_loss: 0.6241 - learning_rate: 3.0000e-04
Epoch 11/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9123 - loss: 0.2547
Epoch 11: val_accuracy improved from 0.79500 to 0.79750, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 11: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 589s 2s/step - accuracy: 0.9101 - loss: 0.2549 - val_accuracy: 0.7975 - val_loss: 0.6381 - learning_rate: 3.0000e-04
Epoch 12/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9156 - loss: 0.2442
Epoch 12: val_accuracy improved from 0.79750 to 0.79850, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 12: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 593s 2s/step - accuracy: 0.9189 - loss: 0.2340 - val_accuracy: 0.7985 - val_loss: 0.6607 - learning_rate: 3.0000e-04
Epoch 13/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9305 - loss: 0.1898
Epoch 13: ReduceLROnPlateau reducing learning rate to 9.000000427477062e-05.

Epoch 13: val_accuracy did not improve from 0.79850
250/250 ━━━━━━━━━━━━━━━━━━━━ 615s 2s/step - accuracy: 0.9295 - loss: 0.1957 - val_accuracy: 0.7980 - val_loss: 0.6897 - learning_rate: 3.0000e-04
Epoch 14/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9323 - loss: 0.1953
Epoch 14: val_accuracy improved from 0.79850 to 0.81350, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 14: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 591s 2s/step - accuracy: 0.9365 - loss: 0.1816 - val_accuracy: 0.8135 - val_loss: 0.6585 - learning_rate: 9.0000e-05
Epoch 15/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9446 - loss: 0.1614
Epoch 15: val_accuracy did not improve from 0.81350
250/250 ━━━━━━━━━━━━━━━━━━━━ 613s 2s/step - accuracy: 0.9465 - loss: 0.1558 - val_accuracy: 0.8085 - val_loss: 0.6749 - learning_rate: 9.0000e-05
Restoring model weights from the end of the best epoch: 14.
✅ Aşama 1 tamamlandı! En iyi val_accuracy: %81.35


In [ ]:
# ── HÜCRE 9: Aşama 2 — Fine-Tuning (Son 50 Katman) ────────────────────────
# InceptionV3'ün son inception bloklarını çok düşük LR ile açıyoruz.

base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_2 = [
    EarlyStopping(patience=4, restore_best_weights=True,
                  monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=2, min_lr=1e-7,
                     monitor='val_loss', verbose=1),
    ModelCheckpoint(kayit_yolu, save_best_only=True,
                    monitor='val_accuracy', verbose=1)
]

print('🔧 Aşama 2 — Fine-Tuning Başlıyor...')
history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_2,
    verbose=1
)
print(f'✅ Fine-Tuning tamamlandı! En iyi val_accuracy: %{max(history_2.history["val_accuracy"])*100:.2f}')

🔧 Aşama 2 — Fine-Tuning Başlıyor...
Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8902 - loss: 0.3457
Epoch 1: val_accuracy improved from None to 0.78250, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 703s 3s/step - accuracy: 0.8955 - loss: 0.3143 - val_accuracy: 0.7825 - val_loss: 0.7392 - learning_rate: 1.0000e-05
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9192 - loss: 0.2329
Epoch 2: val_accuracy improved from 0.78250 to 0.79200, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 690s 3s/step - accuracy: 0.9211 - loss: 0.2267 - val_accuracy: 0.7920 - val_loss: 0.7263 - learning_rate: 1.0000e-05
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9352 - loss: 0.1866
Epoch 3: val_accuracy improved from 0.79200 to 0.80150, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 3: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 688s 3s/step - accuracy: 0.9356 - loss: 0.1852 - val_accuracy: 0.8015 - val_loss: 0.6918 - learning_rate: 1.0000e-05
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9484 - loss: 0.1577
Epoch 4: val_accuracy improved from 0.80150 to 0.81800, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 4: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 687s 3s/step - accuracy: 0.9482 - loss: 0.1589 - val_accuracy: 0.8180 - val_loss: 0.6721 - learning_rate: 1.0000e-05
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9562 - loss: 0.1281
Epoch 5: val_accuracy improved from 0.81800 to 0.81850, saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5



Epoch 5: finished saving model to /content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 684s 3s/step - accuracy: 0.9566 - loss: 0.1296 - val_accuracy: 0.8185 - val_loss: 0.6612 - learning_rate: 1.0000e-05
Epoch 6/15
 66/250 ━━━━━━━━━━━━━━━━━━━━ 6:54 2s/step - accuracy: 0.9681 - loss: 0.0934

In [ ]:
# ── HÜCRE 10: Final Değerlendirme ─────────────────────────────────────────
from tensorflow.keras.models import load_model

best_model = load_model(kayit_yolu)
loss, acc  = best_model.evaluate(val_ds, verbose=0)

print('\n' + '='*50)
print(f'  ✅ InceptionV3 Final Test Doğruluğu: %{acc*100:.2f}')
print(f'  📍 Model kaydedildi: {kayit_yolu}')
print('='*50)